# Walrus Operator `:=` — Assignment Expressions (Python 3.8+)

## What is it?

The walrus operator `:=` lets you **assign a value AND use it in the same expression**.  
It's called a *walrus* because `:=` looks like walrus eyes and tusks.

The core idea: sometimes you compute a value to test it — then need that same value again  
after the test. Without `:=` you write it twice or store it on a separate line first.  
With `:=` you do both in one shot.

```python
# Without — len(data) computed twice:
if len(data) > 10:
    print(f'too long: {len(data)}')

# With walrus — compute once, test it, reuse it:
if (n := len(data)) > 10:
    print(f'too long: {n}')
```

> **Decision rule:** use `:=` when you need a value **both *in* a condition and *after* it**,  
> and a separate assignment line would hurt readability by splitting logic that belongs together.

In [1]:
# Before/after comparison
data = list(range(15))

# Without walrus — len() called twice or awkward pre-assignment:
if len(data) > 10:
    print(f'without walrus — {len(data)} items')

# With walrus — compute once, bind to n, test n, use n:
if (n := len(data)) > 10:
    print(f'with walrus    — {n} items')

without walrus — 15 items
with walrus    — 15 items


## Use case 1 — while loop

The pattern walrus was *designed* for: read/receive until a sentinel value.

```python
# Without walrus — duplicated call:
chunk = file.read(8192)
while chunk:
    process(chunk)
    chunk = file.read(8192)   # ← same call, written twice

# With walrus — read and test in one line:
while chunk := file.read(8192):
    process(chunk)
```

Eliminates the duplicate call and makes the loop condition self-contained at the `while` line.

In [1]:
import io

# Simulate a file with io.BytesIO — same interface as a real file object
payload = b'Python walrus operator demo data. ' * 4
buffer = io.BytesIO(payload)
chunk_size = 30
chunks: list[bytes] = []

while chunk := buffer.read(chunk_size):
    chunks.append(chunk)

print(f'Read {len(chunks)} chunks from {len(payload)} bytes:')
for idx, c in enumerate(chunks, 1):
    print(f'  chunk {idx}: {c!r}')

Read 5 chunks from 136 bytes:
  chunk 1: b'Python walrus operator demo da'
  chunk 2: b'ta. Python walrus operator dem'
  chunk 3: b'o data. Python walrus operator'
  chunk 4: b' demo data. Python walrus oper'
  chunk 5: b'ator demo data. '


## Use case 2 — if with assign + test in one shot

When you want to *try* something and immediately branch on whether it worked,  
without a temporary variable that lingers in the surrounding scope.

```python
# Without walrus:
user = db.get(user_id)
if user is None:
    raise NotFound
user.process()

# With walrus — intent is clearer, variable stays close to its use:
if (user := db.get(user_id)) is None:
    raise NotFound
user.process()
```

In [ ]:
from __future__ import annotations

_db: dict[str, dict] = {
    'alice@example.com': {'id': 1, 'name': 'Alice', 'role': 'admin'},
}
_cache: dict[int, int] = {}


def find_user(email: str) -> dict | None:
    return _db.get(email)


def get_score(user_id: int) -> int:
    return user_id * 42


# Pattern 1: None-check + use in one step
email = 'alice@example.com'
if (user := find_user(email)) is None:
    print('user not found')
else:
    print(f"found: {user['name']} (role={user['role']})")

# Pattern 2: cache — return early on hit, compute and store on miss
user_id = 1
if (cached := _cache.get(user_id)) is not None:
    print(f'cache hit  — score={cached}')
else:
    score = get_score(user_id)
    _cache[user_id] = score
    print(f'computed   — score={score}')

# Second call hits cache
if (cached := _cache.get(user_id)) is not None:
    print(f'cache hit  — score={cached}')

## Use case 3 — comprehension with filter on computed value

When you need to **compute a value, filter by it, and include the computed result**  
(not the original input) in the output — walrus avoids calling the function twice per item.

```python
# Without walrus — expensive() called twice per item:
results = [expensive(x) for x in data if expensive(x) > threshold]

# With walrus — compute once, filter and include the same result:
results = [y for x in data if (y := expensive(x)) > threshold]
```

> **Scope gotcha (see demo below):** `:=` inside a comprehension binds to the **enclosing scope**,  
> not to the comprehension's own scope. The last assigned value of `y` survives the comprehension.

In [2]:
import math


def expensive(x: float) -> float:
    return round(math.sqrt(abs(x - 5)) * 10, 2)


data = list(range(10))
threshold = 15.0

# expensive() called exactly ONCE per x — walrus binds result to y for reuse
results = [y for x in data if (y := expensive(x)) > threshold]
print(f'results (computed values that passed filter): {results}')

# Scope leak — y is still bound in THIS scope after the comprehension finishes
print(f'y after comprehension: {y}')   # last value that passed the filter
print('y leaked to enclosing scope — unlike the iteration variable')

# To verify: walrus y exists here, but the comprehension iteration variable x does NOT.
# x was only ever a local binding inside the comprehension scope.
print(f'y in enclosing scope: {"y" in dir()}')
print(f'x in enclosing scope: {"x" in dir()}')   # False — x is comprehension-local

results (computed values that passed filter): [22.36, 20.0, 17.32, 17.32, 20.0]
y after comprehension: 20.0
y leaked to enclosing scope — unlike the iteration variable
y in enclosing scope: True
x in enclosing scope: False


## Real-world Django / FastAPI patterns

```python
# Django — filter().first() + None-guard in one line
if (user := User.objects.filter(email=request.data['email']).first()) is None:
    return Response({'error': 'user not found'}, status=404)
send_welcome_email(user)   # user in scope, no lingering pre-check variable


# FastAPI — extract and validate a field in one step
@router.post('/items')
async def create_item(data: dict = Body(...)):
    if (name := data.get('name')) is None:
        raise HTTPException(400, 'name is required')
    return {'created': name}


# Celery worker — consume until queue is empty
while batch := queue.dequeue(max_items=100):
    process_batch(batch)


# Redis read-through cache
if (cached := redis.get(cache_key)) is not None:
    return json.loads(cached)
result = db.fetch(query)
redis.setex(cache_key, ttl, json.dumps(result))
return result
```

## Gotchas

### 1. Cannot be a standalone statement

`:=` is an *expression*, not a *statement*. You cannot write it alone on a line.

```python
x := 5      # SyntaxError: invalid syntax
(x := 5)    # valid — expression statement wrapped in parens
x = 5       # just use this when you don't need the value in an expression
```

### 2. Parentheses required inside `if` / `while` conditions

Walrus has very low operator precedence. Without parens, the comparison swallows the wrong operand:

```python
if n := len(data) > 10:    # parsed as: n := (len(data) > 10) — n is True/False!
if (n := len(data)) > 10:  # correct: n gets the length, then > 10 is tested
```

### 3. Walrus in comprehensions leaks to enclosing scope (but iteration variable does not)

```python
results = [y for x in data if (y := f(x)) > 0]
# y is visible here (walrus leaked) — x is NOT (comprehension scope)
```

### 4. Don't use it when a plain assignment is clearer

```python
# Overdone — hurts readability, saves nothing meaningful:
print(y := f(x := input()))

# Better — two clear lines:
x = input()
print(f(x))
```

In [ ]:
# Gotcha 1: cannot be a standalone statement
# x := 5     # uncomment → SyntaxError
(x := 5)     # valid as an expression statement
print(f'x = {x}')

# Gotcha 2: precedence trap — always wrap the walrus expression in parens
data = 'hello world'

if (n_wrong := len(data) > 5):          # WRONG: n_wrong = True (the comparison result)
    print(f'precedence trap: n_wrong = {n_wrong!r} — got bool, not int')

if (n := len(data)) > 5:                # CORRECT: n = 11, then test 11 > 5
    print(f'correct:         n = {n!r}  — got the int we wanted')

# Gotcha 3: scope leak from comprehension
squares_over_10 = [y for x in range(6) if (y := x ** 2) > 10]
print(f'squares over 10: {squares_over_10}')
print(f'y after comprehension: {y}')   # 25 — last value that passed the filter

try:
    print(x_iter)
except NameError:
    _ = [x_iter * 2 for x_iter in range(3)]
    # x_iter is scoped inside the comprehension — NameError if accessed here
    print("x_iter (iteration var): NameError — stays inside comprehension scope")

## Summary

| Pattern | Code form | Saves |
|---|---|---|
| while-read | `while chunk := file.read(n)` | Duplicated read call |
| None-check | `if (x := call()) is None` | Pre-assignment + test split |
| Cache lookup | `if (v := cache.get(k)) is not None` | Double lookup |
| Comprehension filter | `[y for x in data if (y := f(x)) > 0]` | Double call to `f(x)` |

**The one rule:** use `:=` only when the combined assign-and-test is *obviously clearer*  
than a plain assignment on a separate line. Density is not a virtue.

**Interview angle:** the examiner wants to hear two things:  
1. It's an expression, not a statement — can't stand alone, must live inside a larger expression.  
2. Comprehension scope leak — walrus binds to the enclosing scope, the iteration variable does not.